In [1]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


# 🧠 Étape 5 : Modélisation (Machine Learning & Deep Learning) (Squelette Étudiant)

Cette étape correspond au cinquième chapitre du cours. L'objectif est d'implémenter d'une part un modèle de Machine Learning tabulaire (ex: RandomForest) et d'autre part un réseau de neurones convolutif (CNN) sous TensorFlow pour traiter des images ou signaux complexes.

### 1. Préparation de l'environnement

In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras import models, layers

print("TensorFlow version :", tf.__version__)


# chemin racine du projet (IMPORTANT)
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

data_path = os.path.join(
    BASE_DIR,
    "data",
    "processed",
    "cleaned_data_sample.csv"
)

print(" Chemin utilisé :", data_path)

df = pd.read_csv(data_path)
print("✔ Dataset chargé :", df.shape)
df.head()

2026-05-21 15:28:50.912679: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version : 2.16.2
📂 Chemin utilisé : /Users/hobb/Documents/GitHub/aptispace-datascience-projet/data/processed/cleaned_data_sample.csv
✔ Dataset chargé : (1460, 85)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,zone_type,coef_multiplicateur,HouseAge,LogArea
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,2,2008,WD,Normal,208500.0,Standard,1.0,23,7.444833
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,5,2007,WD,Normal,181500.0,Premium,1.2,50,7.141245
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,9,2008,WD,Normal,223500.0,Standard,1.0,25,7.488294
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,2,2006,WD,Abnorml,140000.0,Premium,1.2,111,7.448916
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,12,2008,WD,Normal,250000.0,Luxury,1.5,26,7.695758


### 2. Modélisation Tabulaire (Machine Learning)

**À COMPLÉTER PAR L'ÉTUDIANT :**
Entraînez un modèle d'apprentissage supervisé (ex: forêt aléatoire) sur les caractéristiques extraites de votre jeu de données.

In [ ]:
df_feat = df.copy()

if "HouseAge" not in df_feat.columns and "YearBuilt" in df_feat.columns:
    df_feat["HouseAge"] = 2026 - df_feat["YearBuilt"]
    
features = [
    "GrLivArea",
    "OverallQual",
    "HouseAge",
    "GarageCars",
    "TotalBsmtSF",
    "coef_multiplicateur"
]

target = "SalePrice"

# sécurité si NaN
df_ml = df_feat[features + [target]].dropna()

X = df_ml[features]
y = df_ml[target]

print("✔ Features :", features)
print("✔ Dataset :", X.shape)


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("✔ Modèle entraîné")

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n MÉTRIQUES")
print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 4))


importance = pd.DataFrame({
    "feature": features,
    "importance": rf_model.feature_importances_
}).sort_values(by="importance", ascending=False)

importance

✔ Features : ['GrLivArea', 'OverallQual', 'HouseAge', 'GarageCars', 'TotalBsmtSF', 'coef_multiplicateur']
✔ Dataset : (1460, 6)
✔ Modèle entraîné

📊 MÉTRIQUES
MAE : 17719.1
RMSE: 25714.55
R²  : 0.8899


,feature,importance
1,OverallQual,0.593634
0,GrLivArea,0.188243
4,TotalBsmtSF,0.108668
2,HouseAge,0.060593
3,GarageCars,0.042476
5,coef_multiplicateur,0.006386


### 3. Modélisation Vision / Deep Learning (CNN & TensorFlow)

**À COMPLÉTER PAR L'ÉTUDIANT :**
Pour des motifs complexes (images/signaux), mettez en place un réseau convolutif (Conv2D, Pooling, Dense) pour classifier ou enrichir vos prédictions.

In [ ]:
def generate_dummy_images(num_samples=100):
    images = np.zeros((num_samples, 64, 64, 3), dtype=np.float32)
    labels = np.zeros(num_samples, dtype=np.int32)

    for i in range(num_samples):
        label = np.random.choice([0, 1])
        labels[i] = label

        images[i] = 0.2 + np.random.normal(0, 0.01, (64, 64, 3))

        if label == 1:
            images[i, 10:30, 10:30, 0] = 0.8
        else:
            images[i, 20:40, 20:40, 1] = 0.8

    return images, labels


X_images, y_labels = generate_dummy_images(200)

split = int(0.8 * len(X_images))

X_train_img = X_images[:split]
y_train_img = y_labels[:split]

X_test_img = X_images[split:]
y_test_img = y_labels[split:]

print("✔ Images shape :", X_train_img.shape)




cnn_model = models.Sequential([
    layers.Conv2D(16, (3,3), activation="relu", input_shape=(64,64,3)),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(32, (3,3), activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),

    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

cnn_model.summary()



cnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

cnn_model.fit(
    X_train_img,
    y_train_img,
    epochs=3,
    batch_size=16,
    verbose=1
)

print(" CNN entraîné")


loss, acc = cnn_model.evaluate(X_test_img, y_test_img)

print("\n CNN RESULTS")
print("Loss :", loss)
print("Accuracy :", acc)

✔ Images shape : (160, 64, 64, 3)


/Users/hobb/Documents/GitHub/aptispace-datascience-projet/venv/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       401,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 406,625 (1.55 MB)

 Trainable params: 406,625 (1.55 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.9375 - loss: 0.2857
Epoch 2/3
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 1.0000 - loss: 0.0034
Epoch 3/3
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 1.0000 - loss: 2.2350e-05
🚀 CNN entraîné
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - accuracy: 1.0000 - loss: 4.4233e-06 

📊 CNN RESULTS
Loss : 4.423252903507091e-06
Accuracy : 1.0
